In [ ]:
import pandas as pd
import numpy as np

regions = ["H1", "H2", "L1", "L2", "L3"]
seq_regions = ["SEQ_H1", "SEQ_H2", "SEQ_L1", "SEQ_L2", "SEQ_L3"]
cf_regions = ["CF_H1", "CF_H2", "CF_L1", "CF_L2", "CF_L3"]
len_regions = ["LEN_H1", "LEN_H2", "LEN_L1", "LEN_L2", "LEN_L3"]
meta_cols = ["pdb", "Hchain", "Lchain", "model", "antigen_name", "antigen_species"]


ab_ag_scalop = (
    pd.read_csv("data/ab_ag_scalop.tsv", sep="\t")
    .dropna(subset = seq_regions)
    #.dropna(subset = cf_regions)
    .drop_duplicates(subset = seq_regions) 
)

antigen_counts = ab_ag_scalop["antigen_name"].value_counts() # Tabelle aus antigen_names und ihren Häufigkeiten in der Spalte antigen_name
ab_ag_scalop = ab_ag_scalop[ab_ag_scalop["antigen_name"].isin(antigen_counts[antigen_counts >= 70].index)] # Behält nur Zeilen, deren antigen_name mindestens 5-mal vorkommt

In [ ]:
for cf_region, seq_region in zip(cf_regions, seq_regions):    
    print(f"Cluster in {cf_region}: {sorted(ab_ag_scalop[cf_region].astype(str).unique().tolist())}")
    print(f"Längen in {cf_region}: {sorted(ab_ag_scalop[seq_region].astype(str).map(len).unique().tolist())}\n")
    

In [ ]:
# Before clustering, Scalop groups CDR-sequences by length. However, some sequences in our dataset have a length, for which Scalop cannot assign a cluster.
# These sequences are unusually long or short (e.g. für CDR-H1: lengths 4, 5, 6, 12, 13, 16 do not yield any clusters, lenghts 7 and 8 do yield clusters) and are rare in our dataset as they are outliers.
# Proportion tests will be conducted only on length-groups, for which Scalop can assign clusters. Sequences in those length-groups, which were not assigned a cluster (nan) will be considered as cluster "others"
# in their respective length-group. Length groups, that did not yield any clusters will be discarded and not analyzed further.

In [ ]:
regions_dict = {}

for region, seq_region, cf_region, len_region in zip(regions, seq_regions, cf_regions, len_regions):

    # DataFrame für jede Region, der alle Sequenzen nach Längengruppe und Cluster sortiert

    df = ab_ag_scalop[meta_cols + [seq_region, cf_region]].copy() # Erstellt eue Kopie von df für jede Region, damit nicht jede Region den originalen df überschreibt
    df[len_region] = df[seq_region].str.len() # Neue Spalte für die Sequenzlänge
    df[cf_region] = df[cf_region].fillna("others") # Ersetzt fehlende Cluster durch "others" 
    df = df.sort_values(by=[len_region, cf_region]) # Sortiert nach Länge und Cluster


    # DataFrame für jede Region aufteilen in DataFrames für jede Längengruppe und filtern nach Längengruppen mit gültigen Cluster-Zuordnungen durch Scalop

    valid_lens = (
    df[df[cf_region] != "others"] # nur Zeilen, in denen es gültige Cluster-Zuordnungen durch Scalop gibt
    [len_region].unique().tolist() # Sequenzlängen, die in den gültigen Zeilen vorkommen, aus der Spalte zur Sequenzlänge extrahieren
    )

    len_groups_dict = {} # Dictionary initialisieren für die DataFrames aller gültigen Längen-Gruppen
    for len_group in valid_lens:
        df_len = df[df[len_region] == len_group] # Zeilen aus dem großen DataFrame filtern, die zur gleichen, gültigen Längengruppe gehören
        len_groups_dict[len_group] = df_len # gefilterten DataFrame in das Dictionary aufnehmen

    regions_dict[region] = len_groups_dict # Dictionary für DataFrames derselben Region in ein äußeres Dictionary anlegen

In [ ]:
# Chi2 Goodness of Fit Test für Vergleich zwischen Cluster und gesamten Datensatz ab_ag_scalop.tsv

from scipy.stats import chisquare

total_counts = ab_ag_scalop["antigen_name"].value_counts() # pd.Series (1D-Tabelle): antigen_name (index, keine Spalte) --> abs. Häufigkeit in ab_ag_scalop (Spalte mit Werten))
total_props = total_counts / total_counts.sum() # pd.Series (1D-Tabelle): antigen_name (index, keine Spalte) --> rel. Häufigkeit in ab_ag_scalop (Spalte mit Werten)
all_antigens = total_counts.index # pd.Series (1D-Tabelle): Zahl (index, keine Spalte) --> antigen_name (Spalte mit Werten)

for region in regions:
    for len_group in regions_dict[region]:
        cluster = regions_dict[region][len_group] # wählt den pd.DataFrame einer Längengruppe einer CDR-Region

        cluster_counts_obs = (
            cluster["antigen_name"]
            .value_counts() # pd.Series (1D-Tabelle): antigen_name (index, keine Spalte) --> abs. Häufigkeit im Cluster (Spalte mit Werten)
            .reindex(all_antigens, fill_value = 0)
        )
        # Erklärung zu .reindex(...)
        # Antigene, die im Cluster nicht vorkommen, erscheinen nicht in cluster_counts
        # --> cluster_counts wird erweitert durch die Antigene aus total_counts (stehen im Index von total_counts)
        # Häufikeiten dieser Antigene im Cluster ist 0

        cluster_counts_exp = total_props[all_antigens].values * cluster_counts_obs.sum()
        # Erwartete (bei H0) abs. Häufigkeit im Cluster =  rel. Häufigkeit im ganzen Datensatz * Anzahl aller Counts (nicht Ag-spezifisch) im Cluster

        # ---- Neu: Warnungen für kritische erwartete Counts ----
        critical_expected = cluster_counts_exp < 5
        num_critical = critical_expected.sum()
        total_categories = len(cluster_counts_exp)

        if num_critical > 0:
            print(f"{region} - Länge {len_group}: {num_critical}/{total_categories} Antigene mit erwarteten Counts < 5")

        # Goodness-of-Fit-Test
        chi2_stat, p_value = chisquare(f_obs=cluster_counts_obs, f_exp=cluster_counts_exp)

        print(f"{region} - Länge {len_group}: Chi² = {chi2_stat:.2f}, p = {p_value:.4f}\n")


In [ ]:
fromscipy.stats import chi2_contingency

# Ergebnisse für jede Region und Längengruppe
for region in regions_dict:
    for len_group in regions_dict[region]:

        # Hole Cluster-zuordnung + Antigen
        df = regions_dict[region][len_group]

        # Kontingenztabelle: Cluster (cf_region) × Antigen
        cf_region = f"CF_{region}"
        contingency = pd.crosstab(df[cf_region], df["antigen_name"])

        # Chi²-Test
        chi2_stat, p_value, dof, expected = chi2_contingency(contingency)

        print(f"[{region}] Länge {len_group}: Chi² = {chi2_stat:.2f}, df = {dof}, p = {p_value:.4f}")
